# Work Rate Limited Variable Duration Pendulum Swing Up

The positive work done by the torque input is calculated using a soft plus
function to avoid non-smoothness that would arise from using a standard max.

In this example, the goal is to put a limit on the work that can be done in 2 seconds.


In [ ]:
import os
import numpy as np
import sympy as sm
from opty import Problem
import matplotlib.pyplot as plt
import matplotlib.animation as animation

### Start with defining the fixed duration and number of nodes.

In [ ]:
target_angle = np.pi
num_nodes = 501

### Symbolic equations of motion

$W$ is the total positive work done by the torque input. So negative power or braking is ignored as an extra challenge. (For this particular problem the power seems to never get negative but I will leave this equation in place to potential problems that it could cause) Its derivative is given by the equation:

$$ \frac{d}{dt} W= \frac{1}{s}  \ln(e^{s P(t)} + 1) \approx \max(0, P(t))$$
$$ P(t) = T(t) \omega(t)$$

Which comes with a sharpness factor, $s$, to tune how tight the bend is of the softplus function.

$W_{delay}(t)$ Is a copy of $W$ with a delay of `time_delay` = 2 seconds.

$W_{diff}(t)$ is the amount of energy that is used in the last 2 seconds.
$$W_{diff}(t) = W(t) - W_{delay}(t)$$

In [ ]:
m, g, d, t, h = sm.symbols("m, g, d, t, h", real=True)
theta, omega, T = sm.symbols("theta, omega, T", cls=sm.Function)
W, Wdt, Wdt2 = sm.symbols("W, Wdt, Wdt2", cls=sm.Function)
W_delay, Wdt_delay = sm.symbols("Wdelay, Wdtdelay", cls=sm.Function)
W_diff = sm.symbols("Wdiff", cls=sm.Function)

sharpness = 10.0  # define how tight the softplus function is
# Softplus demo available at: https://www.desmos.com/calculator/sveirxlszn
ddt_W_func = 1 / sharpness * sm.ln(sm.exp(sharpness * (T(t) * omega(t))) + 1)
ddt_W_numeric = sm.lambdify((omega(t), T(t)), ddt_W_func, "numpy")

state_symbols = (theta(t), omega(t), W(t), Wdt(t), W_diff(t), T(t))
constant_symbols = (m, g, d)
specified_symbols = (T(t),)

free_order = [theta(t), omega(t), W(t), Wdt(t), W_diff(t), T(t), Wdt_delay(t), h] # toggle line on for disconnected Wdt but solving
# free_order = [theta(t), omega(t), W(t), Wdt(t), W_diff(t), T(t), h]  # toggle line off for disconnected Wdt but solving

# eom based on first derivatives
eom = sm.Matrix(
    [
        theta(t).diff() - omega(t),
        m * d**2 * omega(t).diff() + m * g * d * sm.sin(theta(t)) - T(t),
        W(t).diff() - ddt_W_func,
        W(t).diff() - Wdt(t),
        W_diff(t).diff() - (ddt_W_func - Wdt_delay(t)),
        # W_diff(t).diff() - (ddt_W_func - W_delay(t).diff()),
    ]
)
eom

Specify the known system parameters.

In [ ]:
par_map = {
    m: 1.0,
    g: 9.81,
    d: 1.0,
}
time_delay = 2.0  # seconds

Known trajectories are used for the calculation of the delayed signals. 

`delaydt_traj(free)`:
In order to calculate the delayed version of the derivative, first the undelayed derivative of $W(t)$ needs to be calculated (`Wdt` is not a state variable and thus not accesible in `free`). The values of torque $T(t)$ and rotational velocity $\omega(t)$ can be grabbed from the free vector for this calculation.

In [ ]:
def W_delay_traj(free):
    time = np.linspace(0, free[-1] * (num_nodes - 1), num_nodes)
    delayed_time = np.clip(time - time_delay, 0, None)
    W_index = free_order.index(W(t))
    W_arr = free[W_index * num_nodes : (W_index + 1) * num_nodes]
    W_delay_arr = np.interp(delayed_time, time, W_arr)
    return W_delay_arr


def Wdt_delay_traj(free):
    time = np.linspace(0, free[-1] * (num_nodes - 1), num_nodes)
    delayed_time = np.clip(time - time_delay, 0, None)
    T_index = free_order.index(T(t))
    omega_index = free_order.index(omega(t))
    T_arr = free[T_index * num_nodes : (T_index + 1) * num_nodes]
    omega_arr = free[omega_index * num_nodes : (omega_index + 1) * num_nodes]
    Wdt_arr = ddt_W_numeric(omega_arr, T_arr)
    Wdt_delay_arr = np.interp(delayed_time, time, Wdt_arr)
    return Wdt_delay_arr


known_traj_map = {
    W_delay(t): W_delay_traj,
    W_delay(t).diff(): Wdt_delay_traj,
    # Wdt_delay(t): Wdt_delay_traj,  # toggle line off for disconnected Wdt but solving
}

Specify the objective function and it's gradient. In this case, make the problem instance the first argument and it is available to use inside the these functions. This shows how to use ``.parse_free()``, ``.extract_values()``, and ``.fill_free()`` to manage the numerical vectors.


In [ ]:
def obj(prob, free):
    """Minimize the sum of the squares of the control torque."""
    T_vals = prob.extract_values(free, T(t))
    h_val = prob.extract_values(free, h)
    return h_val * np.sum(T_vals**2)


def obj_grad(prob, free):
    T_vals = prob.extract_values(free, T(t))
    h_val = prob.extract_values(free, h)
    grad = np.zeros_like(free)
    prob.fill_free(grad, 2.0 * h_val * T_vals, T(t))
    prob.fill_free(grad, np.sum(T_vals**2), h)
    return grad

Specify the symbolic instance constraints, i.e. initial and end conditions using node numbers 0 to N - 1

In [ ]:
instance_constraints = (
    theta(0 * h),
    theta((num_nodes - 1) * h) - target_angle,
    omega(0 * h),
    omega((num_nodes - 1) * h),
    W(0 * h),
    W_diff(0 * h),
)

Specify the variable bounds for each state and input.

In [ ]:
bounds = {
    T(t): (-2.0, 2.0),
    W(t): (0, np.inf),
    W_diff(t): (0, 4.0),
    h: (0.0, 0.5),
}

Create an optimization problem. If the backend is set to ``numpy``, no C compiler is needed and the problem can be solved using pure Python code. There is a large performance loss but for simple problems performance may not be a concern.

In [ ]:
prob = Problem(
    obj,
    obj_grad,
    eom,
    state_symbols,
    num_nodes,
    h,
    known_parameter_map=par_map,
    instance_constraints=instance_constraints,
    time_symbol=t,
    bounds=bounds,
    known_trajectory_map=known_traj_map,
    backend="cython",
    tmp_dir=os.path.join(os.getcwd(), ".opty_bin"),
)
prob.add_option("max_iter", 5000)
prob.add_option("mumps_mem_percent", 16000)
# prob.add_option("derivative_test", "first-order")

In [ ]:
# verify the free order matches extraction indices
if list(prob._extraction_indices.keys()) != free_order:
    print(f"used: {free_order}")
    print(f"Correct: \n{list(prob._extraction_indices.keys())}")
    raise ValueError("Free order does not match extraction indices.")
else:
    print(f"Correct free_order:\n{free_order}")

Use existing solution if available else pick a reasonable initial guess and solve the problem. Use approximately zero as an initial guess to avoid divide-by-zero, and solve the problem.

In [ ]:
script_name = "plot_pendulum_swing_up_variable_duration_power_limit_ipynb"
fname = f"{script_name}_{num_nodes}_nodes_solution.csv"
if os.path.exists(fname):
    prev_solution = np.loadtxt(fname)
    initial_guess = prev_solution[: prob.num_free]
    initial_guess[-1] = prev_solution[-1]  # ensure h is set correctly
else:
    initial_guess = np.full(prob.num_free, 1e-10)

In [ ]:
solution, info = prob.solve(initial_guess)
print(info["status_msg"])
print(info["obj_val"])

In [ ]:
# np.savetxt(fname, solution)

Plot the optimal state and input trajectories.

In [ ]:
T_vals = prob.extract_values(solution, T(t))
omega_vals = prob.extract_values(solution, omega(t))
time_vals = prob.time_vector(solution)
power_vals =  omega_vals * T_vals
power_vals_rect = ddt_W_numeric(omega_vals, T_vals)

num_axes = len(free_order) - 1 + len(known_traj_map) + 2
fig, axes = plt.subplots(num_axes, 1, figsize=(8, 16), sharex=True, layout="compressed")
axes = axes.flatten()

_ = prob.plot_trajectories(solution, axes)

power_ax = axes[-2]
power_ax.plot(time_vals, power_vals)
power_ax.set_ylabel(sm.latex(T(t)*omega(t), mode="inline"))

power_ax2 = axes[-1]
power_ax2.plot(time_vals, power_vals_rect)
power_ax2.set_ylabel(r"$\frac{d}{dt}W =$ " + sm.latex(ddt_W_func, mode="inline"))

for ax in axes:
    ax.grid()
    ax.set_ylabel(ax.get_ylabel(), rotation="horizontal", ha="right")

axes[-1].set_xlabel("Time (s)")

plt.show()

Plot the constraint violations.

In [ ]:
_ = prob.plot_constraint_violations(solution, subplots=True)